In [3]:
import numpy as np
import pandas as pd
from minisom import MiniSom
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [10]:
# Get data into dataframe
df = pd.read_csv('WMA_fractions_v2_with_PV_GMM_ensemble.csv')

: 

In [ ]:
"""
SOM classification of Conservative Temperature, Absolute Salinity and PV,
with several scores (silhouette, Davies-Bouldin, BIC, AIC) compared over
k = 2..10 to help choose the number of classes.

Requires: numpy, pandas, scikit-learn, minisom, matplotlib
Usage (notebook): load your data into a DataFrame called `df`, then run
this script in a cell (paste it, or use %run som_classify_v2.py). It runs on
`df` automatically at the bottom. Change SELECT_BY / N_CLUSTERS below if wanted.
"""
import numpy as np
import pandas as pd
from minisom import MiniSom
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

COLS = ["Conservative_Temperature_[deg_C]", "Absolute_Salinity_[PSU]", "PV"]
K_RANGE = range(2, 11)  # 2..10 inclusive


# --------------------------------------------------------------------------
def suggest_grid(X, nodes_per_sqrt_n=5):
    """Heuristic grid (SOM Toolbox rule of thumb): ~5*sqrt(N) nodes, side
    ratio from the two largest PCA eigenvalues. A rule of thumb only."""
    n_nodes = nodes_per_sqrt_n * np.sqrt(X.shape[0])
    ev = PCA(n_components=2).fit(X).explained_variance_
    ratio = np.sqrt(ev[0] / ev[1])
    ny = max(2, int(round(np.sqrt(n_nodes / ratio))))
    nx = max(2, int(round(n_nodes / ny)))
    return nx, ny


def evaluate_k(W, X, node_ids, k_range, seed=42, cov_type="full"):
    """
    For each k:
      * k-means on the SOM node prototypes (W) -> labels for every sample via
        its best-matching node -> silhouette and Davies-Bouldin on samples X.
      * Gaussian mixture on the node prototypes -> BIC and AIC.
    NOTE: BIC/AIC treat the SOM nodes (not the original samples) as the data
    points, so they depend on the grid size. [Inference]
    """
    rows = []
    for k in k_range:
        if k >= len(W):
            continue
        km = KMeans(n_clusters=k, n_init=10, random_state=seed).fit(W)
        lab = km.labels_[node_ids]
        if len(np.unique(lab)) < 2:
            sil = db = np.nan
        else:
            sil = silhouette_score(X, lab, sample_size=min(5000, len(X)),
                                   random_state=seed)
            db = davies_bouldin_score(X, lab)
        gmm = GaussianMixture(n_components=k, covariance_type=cov_type,
                              n_init=5, random_state=seed).fit(W)
        rows.append(dict(k=k, silhouette=sil, davies_bouldin=db,
                         bic=gmm.bic(W), aic=gmm.aic(W)))
    return pd.DataFrame(rows).set_index("k")


def best_k_per_score(scores):
    return {"silhouette": int(scores["silhouette"].idxmax()),
            "davies_bouldin": int(scores["davies_bouldin"].idxmin()),
            "bic": int(scores["bic"].idxmin()),
            "aic": int(scores["aic"].idxmin())}


def plot_scores(scores, path="cluster_scores.png"):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 4, figsize=(16, 3.5))
    spec = [("silhouette", "higher is better"),
            ("davies_bouldin", "lower is better"),
            ("bic", "lower is better"),
            ("aic", "lower is better")]
    for a, (name, note) in zip(ax, spec):
        a.plot(scores.index, scores[name], "o-")
        a.set_title(f"{name} ({note})")
        a.set_xlabel("k")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


# --------------------------------------------------------------------------
def run_som(df, cols=COLS, grid=None, sigma=None, learning_rate=0.5,
            n_iter=None, k_range=K_RANGE, n_clusters=None,
            select_by="silhouette", cov_type="full", seed=42):
    """
    df          : DataFrame containing `cols`.
    grid        : (nx, ny); if None, suggest_grid() is used.
    n_clusters  : force a number of classes; if None, it is taken from the
                  best k of the `select_by` score
                  ('silhouette', 'davies_bouldin', 'bic' or 'aic').
    Returns (out_df, som, info). info['scores'] holds the score table.
    """
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"Columns not found in DataFrame: {missing}")

    data = df.replace([np.inf, -np.inf], np.nan).dropna(subset=list(cols)).copy()
    n_dropped = len(df) - len(data)

    scaler = StandardScaler()
    X = scaler.fit_transform(data[list(cols)].values)

    nx, ny = grid if grid is not None else suggest_grid(X)
    sigma = sigma if sigma is not None else max(nx, ny) / 2.0
    n_iter = n_iter if n_iter is not None else max(10_000, 500 * nx * ny)

    som = MiniSom(nx, ny, X.shape[1], sigma=sigma, learning_rate=learning_rate,
                  neighborhood_function="gaussian", topology="rectangular",
                  random_seed=seed)
    som.pca_weights_init(X)
    som.train(X, n_iter, random_order=True)

    bmus = np.array([som.winner(x) for x in X])
    data["bmu_x"], data["bmu_y"] = bmus[:, 0], bmus[:, 1]
    data["node_id"] = bmus[:, 0] * ny + bmus[:, 1]
    node_ids = data["node_id"].values

    W = som.get_weights().reshape(nx * ny, -1)  # node prototypes (scaled)

    scores = evaluate_k(W, X, node_ids, k_range, seed, cov_type)
    best = best_k_per_score(scores)

    if n_clusters is None:
        n_clusters = best[select_by]

    # Final grouping of nodes: GMM if chosen by BIC/AIC, otherwise k-means
    if select_by in ("bic", "aic") and n_clusters == best[select_by]:
        node_lab = GaussianMixture(n_components=n_clusters,
                                   covariance_type=cov_type, n_init=5,
                                   random_state=seed).fit(W).predict(W)
    else:
        node_lab = KMeans(n_clusters=n_clusters, n_init=10,
                          random_state=seed).fit(W).labels_
    data["class"] = node_lab[node_ids]

    node_phys = scaler.inverse_transform(W)
    class_centres = pd.DataFrame(
        [node_phys[node_lab == c].mean(axis=0) for c in range(n_clusters)],
        columns=list(cols)).rename_axis("class")

    info = dict(grid=(nx, ny), n_iter=n_iter, sigma=sigma,
                n_dropped_rows=n_dropped, n_clusters=n_clusters,
                select_by=select_by, scores=scores, best_k=best,
                quantization_error=som.quantization_error(X),
                topographic_error=som.topographic_error(X),
                class_centres=class_centres, scaler=scaler)
    return data, som, info


# --------------------------------------------------------------------------
def summarize(out, info, out_csv="classified.csv", plot_path="cluster_scores.png"):
    """Print diagnostics, save the score plot and the classified table."""
    pd.set_option("display.width", 140)
    print(f"Rows dropped (NaN/inf): {info['n_dropped_rows']}")
    print(f"Grid: {info['grid']} | iterations: {info['n_iter']}")
    print(f"Quantization error: {info['quantization_error']:.3f} | "
          f"Topographic error: {info['topographic_error']:.3f}\n")
    print(info["scores"].round(3), "\n")
    print("Best k per score:", info["best_k"])
    print(f"Used k = {info['n_clusters']} (selected by {info['select_by']})\n")
    print(info["class_centres"])
    print(out["class"].value_counts().sort_index())
    plot_scores(info["scores"], plot_path)
    if out_csv:
        out.to_csv(out_csv, index=False)
    print(f"\nSaved {plot_path}" + (f" and {out_csv}" if out_csv else ""))


# --------------------------------------------------------------------------
# Run on the DataFrame `df` that is already loaded in your notebook.
SELECT_BY = "silhouette"   # 'silhouette', 'davies_bouldin', 'bic' or 'aic'
N_CLUSTERS = None          # set an integer to force a number of classes

if "df" in globals():
    out, som, info = run_som(df, select_by=SELECT_BY, n_clusters=N_CLUSTERS)
    summarize(out, info, out_csv=None)   # set out_csv="file.csv" to save
else:
    print("No DataFrame called `df` found; define it first, or call "
          "run_som(your_dataframe) directly.")